# Avaliação posterior nos 45 casos reservados

Use somente depois de concluir o novo treino e fixar os parâmetros. Envie o novo checkpoint e `laya_45_teste_reservado.zip`. Este notebook não treina, calibra ou escolhe limiares. Esses casos foram reservados deste ajuste, mas o corpus já foi examinado em experimentos anteriores.

## 1. Instalar a mesma revisão do Laya

In [ ]:
import os, sys, subprocess
os.environ['USE_TF'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
LAYA_REVISION = 'b8a0f0991d95a953411f2e16f91bddc2ffdaf79a'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
               f'laya @ git+https://github.com/NandhaKishorM/laya.git@{LAYA_REVISION}',
               'transformers==5.22.1', 'scipy', 'accelerate'], check=True)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'não disponível')

## 2. Carregar o checkpoint e o pacote reservado
Você pode enviar um ZIP por vez ou os dois juntos. A célula reaproveita os arquivos já enviados nesta sessão e solicita somente o que falta. Se o checkpoint já estiver no Colab, envie apenas `laya_45_teste_reservado.zip`.

In [ ]:
from pathlib import Path
from google.colab import files
import json, zipfile
WORK = Path('/content/laya_avaliacao_45')
MODEL = WORK / 'model'
TEST = WORK / 'test'
MODEL.mkdir(parents=True, exist_ok=True)
TEST.mkdir(parents=True, exist_ok=True)
loaded = set()

def load_archive(path):
    with zipfile.ZipFile(path) as archive:
        names = set(archive.namelist())
        if 'model.safetensors' in names:
            target, kind = MODEL, 'model'
        elif 'data/holdout.jsonl' in names:
            target, kind = TEST, 'test'
        else:
            raise ValueError('ZIP não reconhecido: ' + str(path))
        if kind in loaded:
            print('Este tipo de pacote já foi carregado:', kind)
            return
        for member in archive.infolist():
            assert (target / member.filename).resolve().is_relative_to(target.resolve()), 'Caminho inválido.'
        archive.extractall(target)
        loaded.add(kind)
        print('Carregado:', path.name)

# Reaproveita os ZIPs enviados numa tentativa anterior, inclusive nomes (1).
for stem in ('laya_semantic_45_45_checkpoint', 'laya_45_teste_reservado'):
    candidates = set()
    for folder in (Path.cwd(), Path('/content')):
        candidates.update(path for path in folder.glob(stem + '*.zip') if path.is_file())
    if candidates:
        load_archive(max(candidates, key=lambda path: path.stat().st_mtime))

while loaded != {'model', 'test'}:
    missing = []
    if 'model' not in loaded: missing.append('laya_semantic_45_45_checkpoint.zip')
    if 'test' not in loaded: missing.append('laya_45_teste_reservado.zip')
    print('Envie somente o que falta:', ', '.join(missing))
    uploads = files.upload()
    if not uploads:
        raise ValueError('Upload cancelado. Execute novamente esta célula para continuar.')
    for name in uploads:
        load_archive(Path(name))
    del uploads
print('Checkpoint e teste reservado prontos. Continue na etapa 3.')

## 3. Avaliar com limiar fixo 0,5

Por padrão, o relatório principal usa apenas estados completos que cabem no contexto e informa a cobertura sobre os 45. Casos longos permanecem identificados como excluídos. `LONG_CONTEXT = True` adiciona escores exploratórios por janelas, separados das métricas de estado completo; isso não garante a comparação conjunta de B/L/R/M. Não escolha essa opção depois de olhar qual produz a melhor métrica.

In [ ]:
LONG_CONTEXT = False
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULT = WORK / 'holdout_45_evaluation.json'
command = [sys.executable, str(TEST / 'evaluate_laya_holdout.py'), '--model', str(MODEL),
           '--data', str(TEST / 'data/holdout.jsonl'), '--out', str(RESULT), '--device', DEVICE]
if LONG_CONTEXT: command.append('--long-context')
subprocess.run(command + ['--dry-run'], check=True)
subprocess.run(command, check=True)
result = json.loads(RESULT.read_text())
print(json.dumps({k:result[k] for k in ('metrics', 'primary_full_state_metrics', 'full_state_coverage_of_45')}, indent=2))

## 4. Baixar o resultado por caso e a curva ROC

In [ ]:
for path in (RESULT, RESULT.with_suffix('.csv'), RESULT.with_name(RESULT.stem + '_roc.csv')):
    files.download(str(path))